# Google Cloud Compute Engine VM 생성 및 Ops Agent 설정

이 노트북은 `gcloud CLI`를 사용하여 GCP Compute Engine 인스턴스를 생성하고, Ops Agent 모니터링 정책을 적용하는 예제입니다.

### 인스턴스 사양 정보
- **프로젝트 ID**: `iceu-songpa03`
- **인스턴스 이름**: `instance-20260914-054747`
- **영역 (Zone)**: `us-central1-a`
- **머신 유형**: `e2-medium`
- **OS 이미지**: `Debian 13 (Trixie)`
- **부트 디스크**: 10GB pd-balanced (인스턴스 삭제 시 자동 삭제 설정)

## 0. 현재 인증 상태 및 프로젝트 확인
실행 전 gcloud에 로그인된 계정과 활성 프로젝트를 확인합니다.

In [ ]:
!gcloud config get-value project
!gcloud auth list

## 1. 원본 요청 명령 실행 (Cross-Platform Python 방식)
Windows, Linux, macOS 환경 모두에서 터미널 개행 문자나 `printf` 명령어 차이 없이 안정적으로 실행되는 원클릭 스크립트입니다.

In [ ]:
import subprocess
import sys

# 1. Compute Engine 인스턴스 생성
print("=== [1/3] Compute Engine 인스턴스 생성 중... ===")
create_vm_cmd = (
    "gcloud compute instances create instance-20260915-143200 "
    "--project=iceu-songpa03 "
    "--zone=us-central1-a "
    "--machine-type=e2-medium "
    "--network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default "
    "--metadata=enable-osconfig=TRUE "
    "--maintenance-policy=MIGRATE "
    "--provisioning-model=STANDARD "
    "--service-account=94943462326-compute@developer.gserviceaccount.com "
    "--scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append "
    "--create-disk=auto-delete=yes,boot=yes,device-name=instance-20260914-054747,disk-resource-policy=projects/iceu-songpa03/regions/us-central1/resourcePolicies/default-schedule-1,image=projects/debian-cloud/global/images/debian-13-trixie-v20260908,mode=rw,size=10,type=pd-balanced "
    "--no-shielded-secure-boot "
    "--shielded-vtpm "
    "--shielded-integrity-monitoring "
    "--labels=goog-ops-agent-policy=v2-template-1-7-0,goog-ec-src=vm_add-gcloud "
    "--reservation-affinity=any"
)
res1 = subprocess.run(create_vm_cmd, shell=True)

if res1.returncode == 0:
    # 2. config.yaml 생성
    print("\n=== [2/3] config.yaml 생성 중... ===")
    config_content = """agentsRule:
  packageState: installed
  version: latest
instanceFilter:
  inclusionLabels:
  - labels:
      goog-ops-agent-policy: v2-template-1-7-0
"""
    with open("config.yaml", "w", encoding="utf-8") as f:
        f.write(config_content)
    print("config.yaml 파일 생성 완료!")

    # 3. Ops Agent 정책 생성
    print("\n=== [3/3] Ops Agent 정책 등록 중... ===")
    policy_cmd = (
        "gcloud compute instances ops-agents policies create goog-ops-agent-v2-template-1-7-0-us-central1-a "
        "--project=iceu-songpa03 "
        "--zone=us-central1-a "
        "--file=config.yaml"
    )
    res2 = subprocess.run(policy_cmd, shell=True)
    print("\n모든 단계가 완료되었습니다.")
else:
    print(f"\n인스턴스 생성 실패 (오류 코드: {res1.returncode})")


## 2. 단계별 실행 (Step-by-Step)
명령어를 단계별로 분리하여 개별 실행하고 결과를 확인할 수 있습니다.

### Step 1: Compute Engine 인스턴스 생성

In [1]:
!gcloud compute instances create instance-20260914-054747 \
    --project=iceu-songpa03 \
    --zone=us-central1-a \
    --machine-type=e2-medium \
    --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default \
    --metadata=enable-osconfig=TRUE \
    --maintenance-policy=MIGRATE \
    --provisioning-model=STANDARD \
    --service-account=94943462326-compute@developer.gserviceaccount.com \
    --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
    --create-disk=auto-delete=yes,boot=yes,device-name=instance-20260914-054747,disk-resource-policy=projects/iceu-songpa03/regions/us-central1/resourcePolicies/default-schedule-1,image=projects/debian-cloud/global/images/debian-13-trixie-v20260908,mode=rw,size=10,type=pd-balanced \
    --no-shielded-secure-boot \
    --shielded-vtpm \
    --shielded-integrity-monitoring \
    --labels=goog-ops-agent-policy=v2-template-1-7-0,goog-ec-src=vm_add-gcloud \
    --reservation-affinity=any

NAME                      ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP    STATUS
instance-20260914-054747  us-central1-a  e2-medium                  10.128.0.5   34.28.117.213  RUNNING


Created [https://www.googleapis.com/compute/v1/projects/iceu-songpa03/zones/us-central1-a/instances/instance-20260914-054747].


### Step 2: config.yaml 파일 생성 (IPython 매직 명령어)

In [2]:
%%writefile config.yaml
agentsRule:
  packageState: installed
  version: latest
instanceFilter:
  inclusionLabels:
  - labels:
      goog-ops-agent-policy: v2-template-1-7-0

Overwriting config.yaml


### Step 3: Ops Agent 정책 등록

In [3]:
!gcloud compute instances ops-agents policies create goog-ops-agent-v2-template-1-7-0-us-central1-a \
    --project=iceu-songpa03 \
    --zone=us-central1-a \
    --file=config.yaml

ERROR: (gcloud.compute.instances.ops-agents.policies.create) ALREADY_EXISTS: Requested entity already exists


## 3. 리소스 생성 결과 확인

In [4]:
# 인스턴스 상태 확인
!gcloud compute instances list --filter="name=instance-20260914-054747"

# Ops Agent 정책 확인
!gcloud compute instances ops-agents policies list --zone=us-central1-a

NAME                      ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP    STATUS
instance-20260914-054747  us-central1-a  e2-medium                  10.128.0.5   34.28.117.213  RUNNING


Listed 0 items.


## 4. (참고) 원본 Bash 스크립트 블록 (Linux / MacOS / WSL 전용)
Linux 또는 Git Bash 커널 환경에서는 아래 `%%bash` 셀로 원본 명령어를 그대로 실행할 수 있습니다.

In [ ]:
%%bash
gcloud compute instances create instance-20260914-054747 \
    --project=iceu-songpa03 \
    --zone=us-central1-a \
    --machine-type=e2-medium \
    --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default \
    --metadata=enable-osconfig=TRUE \
    --maintenance-policy=MIGRATE \
    --provisioning-model=STANDARD \
    --service-account=94943462326-compute@developer.gserviceaccount.com \
    --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
    --create-disk=auto-delete=yes,boot=yes,device-name=instance-20260914-054747,disk-resource-policy=projects/iceu-songpa03/regions/us-central1/resourcePolicies/default-schedule-1,image=projects/debian-cloud/global/images/debian-13-trixie-v20260908,mode=rw,size=10,type=pd-balanced \
    --no-shielded-secure-boot \
    --shielded-vtpm \
    --shielded-integrity-monitoring \
    --labels=goog-ops-agent-policy=v2-template-1-7-0,goog-ec-src=vm_add-gcloud \
    --reservation-affinity=any \
&& \
printf 'agentsRule:\n  packageState: installed\n  version: latest\ninstanceFilter:\n  inclusionLabels:\n  - labels:\n      goog-ops-agent-policy: v2-template-1-7-0\n' > config.yaml \
&& \
gcloud compute instances ops-agents policies create goog-ops-agent-v2-template-1-7-0-us-central1-a \
    --project=iceu-songpa03 \
    --zone=us-central1-a \
    --file=config.yaml

## 5. 인스턴스 및 정책 삭제 (과금 방지 가이드)
실습 종료 후 과금이 발생하지 않도록 생성한 인스턴스와 정책을 삭제합니다.

In [5]:
import subprocess
import json

# 1. 인스턴스 존재 여부 확인 후 안전하게 삭제
print("=== [1/2] Compute Engine 인스턴스 삭제 확인 ===")
check_vm = subprocess.run(
    'gcloud compute instances list --filter="name=instance-20260914-054747" --format=json',
    shell=True, capture_output=True, text=True
)
try:
    vms = json.loads(check_vm.stdout) if check_vm.stdout.strip() else []
except Exception:
    vms = []

if vms:
    print(f"인스턴스 발견: {vms[0].get('name')}, 삭제를 진행합니다...")
    subprocess.run("gcloud compute instances delete instance-20260914-054747 --zone=us-central1-a --quiet", shell=True)
    print("인스턴스 삭제 완료.")
else:
    print("인스턴스(instance-20260914-054747)가 존재하지 않습니다. (이미 삭제되었거나 아직 생성되지 않음 - 정상)")

# 2. Ops Agent 정책 존재 여부 확인 후 안전하게 삭제
print("\n=== [2/2] Ops Agent 정책 삭제 확인 ===")
check_policy = subprocess.run(
    'gcloud compute instances ops-agents policies list --zone=us-central1-a --format=json',
    shell=True, capture_output=True, text=True
)
try:
    policies = json.loads(check_policy.stdout) if check_policy.stdout.strip() else []
except Exception:
    policies = []

if policies:
    print("Ops Agent 정책 발견, 삭제를 진행합니다...")
    subprocess.run("gcloud compute instances ops-agents policies delete goog-ops-agent-v2-template-1-7-0-us-central1-a --zone=us-central1-a --quiet", shell=True)
    print("Ops Agent 정책 삭제 완료.")
else:
    print("등록된 Ops Agent 정책이 존재하지 않습니다. (이미 삭제되었거나 아직 생성되지 않음 - 정상)")

# 3. 잔여 과금 자원 전수 점검 (하네스 실행)
print("\n=== [3/3] 잔여 과금 유발 리소스 점검 ===")
subprocess.run("python .agents/skills/gcp-resource-check/scripts/check_resources.py", shell=True)


=== [1/2] Compute Engine 인스턴스 삭제 확인 ===
인스턴스 발견: instance-20260914-054747, 삭제를 진행합니다...
인스턴스 삭제 완료.

=== [2/2] Ops Agent 정책 삭제 확인 ===
등록된 Ops Agent 정책이 존재하지 않습니다. (이미 삭제되었거나 아직 생성되지 않음 - 정상)

=== [3/3] 잔여 과금 유발 리소스 점검 ===


CompletedProcess(args='python .agents/skills/gcp-resource-check/scripts/check_resources.py', returncode=0)

## 6. 리전별 Compute Engine 비용 비교 및 최저가 Top 3 분석

동일한 구성 사양 기준 전 세계 주요 GCP 리전의 비용을 비교 분석합니다:
- **머신 유형**: `e2-medium` (2 vCPU, 4GB RAM)
- **부트 디스크**: 10GB `pd-balanced`
- **네트워크 인터페이스**: `PREMIUM` 티어 (IPv4 1개 기본 포함)
- **OS 이미지**: `Debian 13` (라이선스 비용 무료)
- **월 기준 시간**: 730시간 (30.4일 24시간 풀가동 기준)

In [ ]:
# GCP 주요 리전별 e2-medium + 10GB pd-balanced 비용 비교 계산기
regions_pricing = [
    # [리전 코드, 리전 명칭, e2-medium 시간당($), pd-balanced GB당 월($)]
    ["us-central1", "미국 아이오와 (Iowa)", 0.033511, 0.10],
    ["us-east1", "미국 사우스캐롤라이나 (S. Carolina)", 0.033511, 0.10],
    ["us-west1", "미국 오리건 (Oregon)", 0.033511, 0.10],
    ["us-east4", "미국 북부 버지니아 (N. Virginia)", 0.036862, 0.11],
    ["us-west2", "미국 로스앤젤레스 (LA)", 0.036862, 0.11],
    ["europe-west1", "유럽 벨기에 (Belgium)", 0.036862, 0.11],
    ["europe-north1", "유럽 핀란드 (Finland)", 0.036862, 0.11],
    ["europe-west4", "유럽 네덜란드 (Netherlands)", 0.038541, 0.11],
    ["asia-east1", "아시아 대만 (Taiwan)", 0.037867, 0.11],
    ["asia-southeast1", "아시아 싱가포르 (Singapore)", 0.040213, 0.12],
    ["asia-northeast3", "아시아 서울 (Seoul)", 0.043564, 0.12],
    ["asia-northeast1", "아시아 도쿄 (Tokyo)", 0.043564, 0.12],
    ["australia-southeast1", "호주 시드니 (Sydney)", 0.046915, 0.13],
    ["southamerica-east1", "남미 상파울루 (Sao Paulo)", 0.052026, 0.15],
]

HOURS_PER_MONTH = 730
DISK_GB = 10
USD_KRW_RATE = 1380  # 기준 환율 (원/달러)

table_data = []
for code, name, vm_hr, disk_gb_mo in regions_pricing:
    vm_mo = vm_hr * HOURS_PER_MONTH
    disk_mo = disk_gb_mo * DISK_GB
    total_mo_usd = vm_mo + disk_mo
    total_hr_usd = total_mo_usd / HOURS_PER_MONTH
    total_mo_krw = total_mo_usd * USD_KRW_RATE
    table_data.append({
        "code": code,
        "name": name,
        "vm_mo": vm_mo,
        "disk_mo": disk_mo,
        "total_mo_usd": total_mo_usd,
        "total_hr_usd": total_hr_usd,
        "total_mo_krw": total_mo_krw
    })

# 총 비용 기준 오름차순 정렬
table_data.sort(key=lambda x: x["total_mo_usd"])
cheapest_cost = table_data[0]["total_mo_usd"]

print("=" * 95)
print(f"{'순위':<4} | {'리전 코드':<20} | {'리전 위치':<26} | {'VM 월비용':<10} | {'디스크':<7} | {'총 월비용(USD)':<12} | {'총 월비용(KRW)'}")
print("-" * 95)

for rank, row in enumerate(table_data, 1):
    diff_pct = ((row['total_mo_usd'] - cheapest_cost) / cheapest_cost) * 100
    diff_str = f"(+{diff_pct:.1f}%)" if diff_pct > 0 else "(최저가)"
    print(f"{rank:<4} | {row['code']:<20} | {row['name']:<26} | ${row['vm_mo']:>7.2f}  | ${row['disk_mo']:>5.2f} | ${row['total_mo_usd']:>6.2f} {diff_str:<7} | 약 {int(row['total_mo_krw']):,}원")

print("=" * 95)
print("\n🏆 [가장 저렴한 리전 Top 3 결과]")
for i in range(3):
    r = table_data[i]
    print(f"  {i+1}위: {r['code']} ({r['name']}) -> 월 ${r['total_mo_usd']:.2f} (시간당 ${r['total_hr_usd']:.5f}, 약 {int(r['total_mo_krw']):,}원/월)")

seoul = next((r for r in table_data if r['code'] == 'asia-northeast3'), None)
if seoul:
    diff_seoul = seoul['total_mo_usd'] - cheapest_cost
    print(f"\n💡 참고: 서울 리전(asia-northeast3)은 월 ${seoul['total_mo_usd']:.2f}로 최저가 리전 대비 월 ${diff_seoul:.2f}(+{(diff_seoul/cheapest_cost)*100:.1f}%) 더 발생합니다.")


### 7. 최저가 리전 선택 및 VM 인스턴스 생성 실행

> [!NOTE]
> 분석 결과, **`us-central1` (현재 설정 리전), `us-east1`, `us-west1`** 3개 리전이 **월 $25.46**으로 전 세계에서 가장 저렴합니다.
>
> 기존에 설정되어 있던 `us-central1-a`가 이미 최저가 리전이므로 그대로 사용하시거나, 다른 최저가 리전인 `us-east1-b` 또는 `us-west1-b`를 선택하여 생성하실 수 있습니다.

In [ ]:
# 원하는 최저가 리전/영역 선택 (us-central1-a, us-east1-b, us-west1-b 중 택1)
TARGET_ZONE = "us-central1-a"  # 또는 'us-east1-b', 'us-west1-b'
TARGET_PROJECT = "iceu-songpa03"
INSTANCE_NAME = "instance-cheapest-vm"

cmd = f"""gcloud compute instances create {INSTANCE_NAME} \
    --project={TARGET_PROJECT} \
    --zone={TARGET_ZONE} \
    --machine-type=e2-medium \
    --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default \
    --metadata=enable-osconfig=TRUE \
    --maintenance-policy=MIGRATE \
    --provisioning-model=STANDARD \
    --create-disk=auto-delete=yes,boot=yes,image=projects/debian-cloud/global/images/debian-13-trixie-v20260908,size=10,type=pd-balanced \
    --no-shielded-secure-boot \
    --shielded-vtpm \
    --shielded-integrity-monitoring
"""

print(f"선택된 리전 영역: {TARGET_ZONE}")
print(f"인스턴스 생성 명령어 실행 준비 완료:\n{cmd}")
# 실행하려면 아래 주석을 해제하세요:
# !{cmd}
